<a href="https://colab.research.google.com/github/mdsadaqathali/week6/blob/main/w6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM,Dense
from tensorflow.keras.callbacks import EarlyStopping
from prophet import Prophet

df=pd.read_csv("covid_19_india.csv")
print(df.head())
print(df.shape)
print(df.columns)
print(df.isnull().sum())

df["Date"]=pd.to_datetime(df["Date"],errors="coerce")
df["Confirmed"]=pd.to_numeric(df["Confirmed"],errors="coerce")

df=df.groupby("Date")["Confirmed"].sum().reset_index()
df=df.sort_values("Date")

df["Daily Confirmed"]=df["Confirmed"].diff()
df["Daily Confirmed"]=df["Daily Confirmed"].fillna(0)
df["Daily Confirmed"]=df["Daily Confirmed"].clip(lower=0)

print(df.head())

plt.figure(figsize=(12,5))
plt.plot(df["Date"],df["Daily Confirmed"])
plt.xlabel("Date")
plt.ylabel("Daily New Cases")
plt.title("COVID-19 Daily New Cases in India")
plt.show()

df=df[["Date","Daily Confirmed"]]
df=df.rename(columns={"Date":"date","Daily Confirmed":"cases"})
df=df.set_index("date")
df["cases"]=pd.to_numeric(df["cases"],errors="coerce")
df["cases"]=df["cases"].fillna(0)

rolling=df["cases"].rolling(7).mean()

plt.figure(figsize=(12,5))
plt.plot(df.index,df["cases"],alpha=0.4,label="Daily Cases")
plt.plot(df.index,rolling,label="7-Day Rolling Average")
plt.xlabel("Date")
plt.ylabel("Cases")
plt.title("COVID-19 Cases with 7-Day Rolling Average")
plt.legend()
plt.show()

result=adfuller(rolling.dropna())
print("ADF Statistic:",result[0])
print("ADF p-value:",result[1])

if result[1]>0.05:
    df["diff"]=df["cases"].diff()
    result2=adfuller(df["diff"].dropna())
    print("Differenced ADF Statistic:",result2[0])
    print("Differenced ADF p-value:",result2[1])

train_size=int(len(df)*0.8)
train=df["cases"][:train_size]
test=df["cases"][train_size:]

arima_model=ARIMA(train,order=(5,1,2))
arima_fit=arima_model.fit()
arima_forecast=arima_fit.forecast(steps=len(test))

arima_mape=mean_absolute_percentage_error(test,arima_forecast)*100
print("ARIMA MAPE:",arima_mape)

prophet_df=df.reset_index()
prophet_df=prophet_df.rename(columns={"date":"ds","cases":"y"})

prophet_train=prophet_df.iloc[:train_size]

prophet_model=Prophet()
prophet_model.fit(prophet_train)

future=prophet_model.make_future_dataframe(periods=len(test))
prophet_forecast=prophet_model.predict(future)

prophet_predictions=prophet_forecast["yhat"].iloc[train_size:].values[:len(test)]

prophet_mape=mean_absolute_percentage_error(test,prophet_predictions)*100
print("Prophet MAPE:",prophet_mape)

scaler=MinMaxScaler()
scaled_data=scaler.fit_transform(df[["cases"]])

window_size=30
forecast_days=7

X=[]
y=[]

for i in range(window_size,len(scaled_data)-forecast_days+1):
    X.append(scaled_data[i-window_size:i,0])
    y.append(scaled_data[i:i+forecast_days,0])

X=np.array(X)
y=np.array(y)

split=int(len(X)*0.8)

X_train=X[:split]
X_test=X[split:]
y_train=y[:split]
y_test=y[split:]

X_train=X_train.reshape((X_train.shape[0],X_train.shape[1],1))
X_test=X_test.reshape((X_test.shape[0],X_test.shape[1],1))

lstm_model=Sequential()
lstm_model.add(LSTM(64,return_sequences=True,input_shape=(window_size,1)))
lstm_model.add(LSTM(32))
lstm_model.add(Dense(7))

lstm_model.compile(optimizer="adam",loss="mse")

early_stopping=EarlyStopping(monitor="val_loss",patience=2,restore_best_weights=True)

history=lstm_model.fit(X_train,y_train,epochs=10,batch_size=32,validation_split=0.2,callbacks=[early_stopping])

lstm_predictions=lstm_model.predict(X_test)

lstm_predictions=lstm_predictions.reshape(-1,1)
lstm_predictions=scaler.inverse_transform(lstm_predictions)
lstm_predictions=lstm_predictions.reshape(-1,7)

actual_lstm=y_test.reshape(-1,1)
actual_lstm=scaler.inverse_transform(actual_lstm)
actual_lstm=actual_lstm.reshape(-1,7)

lstm_mape=mean_absolute_percentage_error(actual_lstm,lstm_predictions)*100
print("LSTM MAPE:",lstm_mape)

plt.figure(figsize=(12,6))
plt.plot(test.index,test.values,label="Actual")
plt.plot(test.index,arima_forecast.values,label="ARIMA")
plt.plot(test.index,prophet_predictions,label="Prophet")
plt.xlabel("Date")
plt.ylabel("Cases")
plt.title("COVID-19 Forecast Comparison")
plt.legend()
plt.show()

print("\nMODEL COMPARISON")
print("ARIMA MAPE:",arima_mape)
print("Prophet MAPE:",prophet_mape)
print("LSTM MAPE:",lstm_mape)

models={"ARIMA":arima_mape,"Prophet":prophet_mape,"LSTM":lstm_mape}
best_model=min(models,key=models.get)

print("Best Model:",best_model)

recent_average=df["cases"].tail(7).mean()

last_window=scaled_data[-window_size:]
last_window=last_window.reshape(1,window_size,1)

future_7=lstm_model.predict(last_window)
future_7=scaler.inverse_transform(future_7.reshape(-1,1)).ravel()

forecast_average=future_7.mean()

increase=((forecast_average-recent_average)/recent_average)*100

print("Current 7-Day Average:",recent_average)
print("Forecast 7-Day Average:",forecast_average)
print("Expected Increase:",increase,"%")

if increase>=50:
    print("SURGE ALERT")
else:
    print("No Surge Alert")
